In [ ]:
# ETL conectado ao Trello para extrair dados de cards e calcular lead time

import requests
import pandas as pd
from datetime import datetime

# =========================
# CONFIGURAÇÕES
# =========================

NOME_QUADRO = "CTI "

# =========================
# FUNÇÃO DE CONEXÃO
# =========================

def trello_get(endpoint, params=None):

    base_url = "https://api.trello.com/1"
    full_url = f"{base_url}/{endpoint}"

    auth = {
        
        
    }

    if params:
        auth.update(params)

    response = requests.get(full_url, params=auth)
    response.raise_for_status()

    return response.json()


# =========================
# BUSCAR QUADRO
# =========================

boards = trello_get("members/me/boards")

board = next(
    (b for b in boards if b['name'] == NOME_QUADRO),
    None
)

if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")


# =========================
# BUSCAR CARDS
# =========================

cards = trello_get(f"boards/{board['id']}/cards")

print(f"✅ {len(cards)} cards encontrados")


# =========================
# ETL - EXTRAÇÃO DAS AÇÕES
# =========================

acoes = []

lead_times = []

for card in cards:

    # Busca TODAS as ações do card
    card_actions = trello_get(
        f"cards/{card['id']}/actions",
        params={'filter': 'all'}
    )

    # -------------------------
    # DATA DE CRIAÇÃO REAL
    # -------------------------

    create_action = next(
        (
            act for act in card_actions
            if act['type'] == 'createCard'
        ),
        None
    )

    data_criacao = None

    if create_action:

        data_criacao = pd.to_datetime(
            create_action['date'],
            errors='coerce'
        )

    # -------------------------
    # DATA ÚLTIMA ATIVIDADE
    # -------------------------

    data_ultima_atividade = pd.to_datetime(
        card.get('dateLastActivity'),
        errors='coerce'
    )

    # -------------------------
    # EXTRAIR MOVIMENTAÇÕES
    # -------------------------

    for act in card_actions:

        if act['type'] == 'updateCard':

            acoes.append({
                'card_id': card['id'],
                'card_name': card['name'],

                'action_date': pd.to_datetime(
                    act['date'],
                    errors='coerce'
                ),

                'list_before': act['data'].get(
                    'listBefore',
                    {}
                ).get('name'),

                'list_after': act['data'].get(
                    'listAfter',
                    {}
                ).get('name'),

                'action_type': act['type']
            })

    # -------------------------
    # LEAD TIME
    # -------------------------

    mov_concluido = [
        act for act in card_actions
        if (
            act['type'] == 'updateCard'
            and
            act['data'].get('listAfter', {}).get('name') == 'Concluido'
        )
    ]

    if mov_concluido:

        # pega a ÚLTIMA vez que entrou em concluído
        ultima_conclusao = mov_concluido[-1]

        data_conclusao = pd.to_datetime(
            ultima_conclusao['date'],
            errors='coerce'
        )

        lead_time_dias = (
            data_conclusao - data_criacao
        ).days

        status = 'Concluído'

    else:

        data_conclusao = None
        lead_time_dias = None
        status = 'Não concluído'

    lead_times.append({

        'card_id': card['id'],
        'card_name': card['name'],

        'data_criacao': data_criacao,

        'data_ultima_atividade': data_ultima_atividade,

        'data_conclusao': data_conclusao,

        'lead_time_dias': lead_time_dias,

        'status': status
    })


# =========================
# DATAFRAMES
# =========================

df_acoes = pd.DataFrame(acoes)

df_lead = pd.DataFrame(lead_times)

df_cards = pd.DataFrame(cards)

print(f"✅ {len(df_acoes)} ações registradas")


# =========================
# CONVERSÃO DATETIME BR
# =========================

# AÇÕES
if not df_acoes.empty:

    df_acoes['action_date'] = pd.to_datetime(
        df_acoes['action_date'],
        errors='coerce'
    ).dt.strftime('%d/%m/%Y %H:%M:%S')


# LEAD TIME
if not df_lead.empty:

    colunas_data_lead = [
        'data_criacao',
        'data_ultima_atividade',
        'data_conclusao'
    ]

    for coluna in colunas_data_lead:

        df_lead[coluna] = pd.to_datetime(
            df_lead[coluna],
            errors='coerce'
        ).dt.strftime('%d/%m/%Y %H:%M:%S')


# CARDS
if not df_cards.empty:

    colunas_datas_cards = [
        'dateLastActivity',
        'start',
        'due'
    ]

    for coluna in colunas_datas_cards:

        if coluna in df_cards.columns:

            df_cards[coluna] = pd.to_datetime(
                df_cards[coluna],
                errors='coerce'
            ).dt.strftime('%d/%m/%Y %H:%M:%S')


# =========================
# EXPORTAÇÃO CSV
# =========================

df_cards.to_csv(
    'cards_trello.csv',
    index=False,
    sep=';'
)

df_acoes.to_csv(
    'acoes_trello.csv',
    index=False,
    sep=';'
)

df_lead.to_csv(
    'lead_time.csv',
    index=False,
    sep=';'
)


# =========================
# LOGS
# =========================

print("\n✅ Arquivos gerados:")

print("  - cards_trello.csv")

print("  - acoes_trello.csv")

print("  - lead_time.csv")

print("\n📊 Amostra do lead_time:")

print(
    df_lead[
        [
            'card_name',
            'data_criacao',
            'data_ultima_atividade',
            'lead_time_dias',
            'status'
        ]
    ].head(10)
)

✅ Conectado: CTI  (ID: 69fd20465fb2b7d1fb85e050)
✅ 20 cards encontrados
✅ 106 ações registradas

✅ Arquivos gerados:
  - cards_trello.csv
  - acoes_trello.csv
  - lead_time.csv

📊 Amostra do lead_time:
                card_name         data_criacao data_ultima_atividade  \
0                 ticket:  08/05/2026 22:14:39   11/05/2026 22:14:52   
1      Consultor: Janaina  08/05/2026 23:08:06   11/05/2026 22:14:51   
2   Documento de abertura  08/05/2026 23:08:12   11/05/2026 21:59:16   
3  Analista: Tiago Madrid  08/05/2026 23:07:59   11/05/2026 22:04:11   
4  Documento de conclusão  08/05/2026 23:09:16   11/05/2026 22:04:12   
5             Alinhamento  08/05/2026 23:09:29   08/05/2026 23:12:26   
6                Kick-Off  08/05/2026 23:09:38   08/05/2026 23:12:54   
7            Configuração  08/05/2026 23:09:45   08/05/2026 23:13:10   
8            Double Check  08/05/2026 23:09:52   08/05/2026 23:16:22   
9  Encaminhar dispositivo  08/05/2026 23:10:02   08/05/2026 23:16:38   

   le